# MOP target selection with Rubin coverage

This notebook selects visible MOP targets, queries their coverage in the selected Rubin Data Release, and generates tables, maps, and individual reports. Run the sections in order.


## 1. Configuration

Choose the Data Release, dates, observatory, and output directory. MOP, TAP, and Butler connections are created automatically when the pipeline runs.


In [ ]:
from importlib import reload
from pathlib import Path

import pandas as pd
import target_selection_pipeline as pipeline

# Reload local changes when rerunning the notebook in the same kernel.
reload(pipeline)
run_target_selection = pipeline.run_target_selection

DATA_RELEASE_NAME = "DP2"  # DP0.1, DP0.2, DP1, or DP2
START_DATE = "2026-08-01"
END_DATE = "2026-08-15"
OUTPUT_DIR = Path("outputs")
OBSERVATORY = "El Leoncito"


## 2. Run the pipeline

Each MOP page is downloaded once with limited concurrency and provides both parameters and photometry. CSV files, MOP pages, 404 responses, photometry, Rubin coverage queries, and plots are cached; the cell reports progress by stage. Use `reuse_cache=False` only to force fresh data.


In [ ]:
combined, paths = run_target_selection(
    start_date=START_DATE,
    end_date=END_DATE,
    data_release=DATA_RELEASE_NAME,
    root_dir=OUTPUT_DIR,
    observatory=OBSERVATORY,
    max_workers=4,
    reuse_cache=True,
    overwrite_target_plots=False,
    verbose=True,
)

print(f"Targets processed: {len(combined)}")
print(f"Products saved to: {paths['run'].resolve()}")
combined.head()


## 3. Explore the results

The following cells display the main products without manually browsing the output directories.


In [ ]:
from IPython.display import Image, Markdown, display
target_summary = pd.read_csv(paths["tables"] / "target_summary.csv")
n_visible = len(target_summary)
n_coverage = int(target_summary["matched_release"].fillna(False).sum())
n_photometry = int(target_summary["mop_photometry_points"].fillna(0).gt(0).sum())
n_both = int(target_summary["matched_with_photometry"].fillna(False).sum())
display(Markdown(
    f"### Run summary\n"
    f"- **Data Release:** {DATA_RELEASE_NAME}\n"
    f"- **Visible targets:** {n_visible}\n"
    f"- **With {DATA_RELEASE_NAME} coverage:** {n_coverage}\n"
    f"- **With MOP photometry:** {n_photometry}\n"
    f"- **With coverage and photometry:** {n_both}\n"
    f"- **Directory:** {paths['run'].resolve()}"
))
summary_columns = [
    "Target", "priority", "release_n_visits",
    *[column for column in target_summary if column.startswith("n_visits_")],
    "mop_photometry_points", "t_E_days", "t_0_HJD", "u_0",
]
display(target_summary[[column for column in summary_columns if column in target_summary]].head(20))


### Complete summary table

The PNG contains every target, visits by filter, and the main MOP parameters. The equivalent CSV preserves the full values.


In [ ]:
summary_png = paths["tables"] / "target_summary.png"
display(Image(filename=str(summary_png)))
print(f"Complete CSV: {(paths['tables'] / 'target_summary.csv').resolve()}")


### Sky maps


In [ ]:
map_paths = [
    paths["sky_plots"] / "sky_by_mag_and_visits.png",
    paths["sky_plots"] / "sky_bulge_zoom_mag_and_visits.png",
]
for map_path in map_paths:
    if map_path.exists():
        display(Markdown(f"#### {map_path.stem.replace('_', ' ')}"))
        display(Image(filename=str(map_path), width=1100))


### Individual report examples

Only three reports are displayed by default to keep the notebook compact. Change `MAX_REPORTS_TO_DISPLAY` to display more.


In [ ]:
MAX_REPORTS_TO_DISPLAY = 3
report_paths = sorted(paths["targets"].glob("*_target_report.png"))
print(f"Available reports: {len(report_paths)}")
for report_path in report_paths[:MAX_REPORTS_TO_DISPLAY]:
    target_name = report_path.name.removesuffix("_target_report.png")
    display(Markdown(f"#### {target_name}"))
    display(Image(filename=str(report_path), width=1300))


## 4. Technical validation

Checks that MOP parameters are present in the tables and shows where to find the products.


In [ ]:
summary_path = paths["tables"] / "visible_summary.csv"
combined_path = paths["tables"] / "combined_targets.csv"
summary_csv = pd.read_csv(summary_path)
combined_csv = pd.read_csv(combined_path)
parameter_columns = [
    column for column in combined.columns
    if column.startswith("mop_")
    and column not in {"mop_link", "mop_parameters_status", "mop_parameters_error"}
]
missing_summary = sorted(set(parameter_columns) - set(summary_csv.columns))
missing_combined = sorted(set(parameter_columns) - set(combined_csv.columns))
if missing_summary or missing_combined:
    raise RuntimeError(f"Missing parameters in CSV files: summary={missing_summary}, combined={missing_combined}")
status_counts = (
    combined["mop_parameters_status"].value_counts(dropna=False).to_dict()
    if "mop_parameters_status" in combined else {}
)
print(f"MOP page status: {status_counts}")
print(f"Measured parameters: {parameter_columns or 'none'}")
print(f"Reports with coadds: {len(report_paths)}")
print(f"Final table: {combined_path.resolve()}")
print(f"Visual summary: {summary_png.resolve()}")
